# Fetch golden-fixture JSON — Phase 2 (Work item E)

Fetch sanitized view-details JSON for the 7 views chosen by `select_golden_fixtures.ipynb` (6 query-selected + 1 hand-picked `custom_tab` fixture).

**Run this once the LANL Enterprise Information Applications outage is over.**

Requires:
- `fixtures/selection.json` (from Phase 1)
- the same six env vars as the Step 0 probe: `AUTH_FLOW_CLIENT_ID`, `AUTH_FLOW_CLIENT_SECRET`, `REDIRECT_URI`, `AUTH_URL`, `TOKEN_URL`, `SCOPE`
- your working **`OAuthManager`** class (the Step 0 version, with `private_mode=True`) — paste it into the marked cell below, or run this notebook in the same kernel/session where `OAuthManager` is already defined.

In [1]:
import json
import os
import re
import time

## Configuration

**HOST NOTE (updated 2026-07-28):** the `datacatalog-d.lanl.gov` alias appears RESTORED — during the v12 PROD run the OAuth redirect JSP was served from `datacatalog-d` again. `BASE_URL` below is therefore reverted to the canonical dev host. If the sanity-check cell fails with connection/4xx errors, fall back to the 7/27 temporary host `https://datacatalog-b.lanl.gov/denodo-data-catalog` and re-run.

In [2]:
BASE_URL = "https://datacatalog-d.lanl.gov/denodo-data-catalog"  # canonical dev host (alias restored 2026-07-28)
# Fallback if -d fails: "https://datacatalog-b.lanl.gov/denodo-data-catalog"
VIEW_DETAILS_PATH = "/public/api/view-details"
SERVER_ID = 1
SOURCE_ENV = "dev"  # content is the dev catalog regardless of the temp hostname
SELECTION_PATH = "fixtures/selection.json"
OUT_DIR = "fixtures"
SLEEP_BETWEEN_CALLS = 0.15

## Sanity check — confirm the host answers before fetching

One cheap `view-details` call against a known-good view. If this fails, switch `BASE_URL` to the `-b` fallback (see HOST NOTE) and re-run.

In [3]:
# Run AFTER auth_manager.authenticate() below if you prefer; placed here as a reference.
def sanity_check(auth_manager):
    r = auth_manager.get(
        BASE_URL + VIEW_DETAILS_PATH,
        params={"viewName": "announcement", "databaseName": "dataportal", "serverId": SERVER_ID},
    )
    print("view-details:", r.status_code)
    assert r.status_code == 200, "Host not answering as expected -- switch BASE_URL to the -b fallback."
    print("Sanity check passed -- OK to run main().")


## Sanitizer

Redacts person-identifying strings; keeps technical content intact.

**2026-07-27 finding:** `connectionUris` JDBC URLs embed the calling user's ID (e.g. `jdbc:vdb://...?user=414515&password=...`). The uid is redacted; the password is already a placeholder.

In [4]:
def sanitize(obj):
    """Redact person-identifying strings; keep technical content intact."""
    if isinstance(obj, dict):
        return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize(v) for v in obj]
    if isinstance(obj, str):
        s = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "REDACTED_EMAIL", obj)
        s = re.sub(r"https?://pbplus\.lanl\.gov[^\s\"'<>]*", "REDACTED_PERSON_URL", s)
        s = re.sub(r"([?&;]user=)[^&;\"'<>\s]+", r"\1REDACTED_UID", s, flags=re.IGNORECASE)
        s = re.sub(r"(UID=)[^;\"'<>\s]+", r"\1REDACTED_UID", s, flags=re.IGNORECASE)
        return s
    return obj

## Fetch function

In [5]:
def main(auth_manager):
    assert os.path.exists(SELECTION_PATH), (
        f"{SELECTION_PATH} not found — run select_golden_fixtures.ipynb first."
    )
    with open(SELECTION_PATH) as f:
        selection = json.load(f)

    fetched_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    for label, sel in selection.items():
        view, db = sel["view_name"], sel["db"]
        resp = auth_manager.get(
            BASE_URL + VIEW_DETAILS_PATH,
            params={"viewName": view, "databaseName": db, "serverId": SERVER_ID},
        )
        record = {
            "_fixture_label": label,
            "_view_name": view,
            "_db_name": db,
            "_http_status": resp.status_code,
            "_fetched_at": fetched_at,
            "_source_env": SOURCE_ENV,
        }
        if resp.status_code == 200:
            record["response"] = sanitize(resp.json())
        else:
            # error_500 fixture: the failure itself IS the fixture content
            record["response"] = None
            record["_error_body"] = resp.text[:500]
        path = os.path.join(OUT_DIR, f"{label}.json")
        with open(path, "w") as f:
            json.dump(record, f, indent=2)
        print(f"[{label}] {view} -> HTTP {resp.status_code} -> {path}")
        time.sleep(SLEEP_BETWEEN_CALLS)

    print("\nDone. Next: hand-write expected/<label>.py per the "
          "expected_plain_example.py template, then run contract_validator.py.")

## Paste your `OAuthManager` here

Paste the working Step 0 `OAuthManager` class (with `private_mode=True`) into the cell below — or skip it if this notebook is running in a kernel where `OAuthManager` is already defined.

In [6]:
import html
import json
import os
import re
import time

import requests
import truststore
import webview
from requests_oauthlib import OAuth2Session

truststore.inject_into_ssl()  # handles LANL internal TLS certs

required = ["AUTH_FLOW_CLIENT_ID", "AUTH_FLOW_CLIENT_SECRET",
            "REDIRECT_URI", "AUTH_URL", "TOKEN_URL", "SCOPE"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}")
print("All 6 env vars present \u2713")


class OAuthManager:
    """Complete OAuth manager that handles initial auth AND refresh"""

    def __init__(self, client_id, client_secret, redirect_uri, auth_url, token_url, scope):
        self.client_id = client_id
        self.client_secret = client_secret
        self.redirect_uri = redirect_uri
        self.auth_url = auth_url
        self.token_url = token_url
        self.scope = scope
        self.oauth = None
        self.token = None
        self.access_token = None
        self.refresh_token = None
        self.token_expiry = None

    def authenticate(self):
        self.oauth = OAuth2Session(self.client_id, redirect_uri=self.redirect_uri, scope=self.scope)
        authorization_url, state = self.oauth.authorization_url(self.auth_url)
        print("Authenticating...")
        print(f"[DEBUG] authorization_url = {authorization_url}")
        authorization_response = self._launch_browser_auth(authorization_url)
        if not authorization_response:
            print("\u2717 Authentication failed")
            return False
        self.token = self.oauth.fetch_token(
            self.token_url, authorization_response=authorization_response,
            client_secret=self.client_secret,
        )
        self.access_token = self.token["access_token"]
        self.refresh_token = self.token.get("refresh_token")
        self.token_expiry = time.time() + self.token.get("expires_in", 3600)
        print("\u2713 Authentication successful")
        return True

    def _launch_browser_auth(self, authorization_url):
        authorization_response = None

        def on_loaded():
            nonlocal authorization_response
            current_url = window.get_current_url()
            print(f"[DEBUG] page loaded: {current_url}")
            if current_url and self.redirect_uri in current_url:
                authorization_response = current_url
                print("\u2713 Captured Authorization")
                window.hide()
                time.sleep(2.5)
                window.destroy()

        window = webview.create_window(
            "OAuth Authorization", authorization_url, width=800, height=600,
            resizable=True, on_top=True,
        )
        window.events.loaded += on_loaded
        webview.start(private_mode=True)  # never reuse a cached LANL SSO session
        return authorization_response

    def _is_token_expired(self):
        if not self.token_expiry:
            return True
        return time.time() >= (self.token_expiry - 60)

    def _refresh_access_token(self):
        if not self.refresh_token:
            print("\u26a0 No refresh token available, re-authenticating...")
            return self.authenticate()
        try:
            new_token = self.oauth.refresh_token(
                self.token_url, refresh_token=self.refresh_token,
                client_id=self.client_id, client_secret=self.client_secret,
            )
            self.token = new_token
            self.access_token = new_token["access_token"]
            self.refresh_token = new_token.get("refresh_token", self.refresh_token)
            self.token_expiry = time.time() + new_token.get("expires_in", 3600)
            print("\u2713 Token refreshed successfully")
            return True
        except Exception as e:
            print(f"\u2717 Refresh failed: {e}")
            print("Re-authenticating...")
            return self.authenticate()

    def _ensure_authenticated(self):
        if not self.access_token or self._is_token_expired():
            if self.refresh_token:
                self._refresh_access_token()
            else:
                self.authenticate()

    def get(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.get(url, headers=headers, **kwargs)

    def post(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.post(url, headers=headers, **kwargs)


# ---------------- PROD configuration ----------------
BASE_URL = "https://datacatalog.lanl.gov/denodo-data-catalog"  # PRODUCTION
TARGET_DB = "dataportal"
SERVER_ID = 1            # confirm unchanged on PROD post-migration
ELEMENT_TYPE = "VIEWS"   # confirmed via Swagger on 2026-07-27
MAX_VIEWS_PER_GROUP = 3  # widened sample vs. the -b run

auth_manager = OAuthManager(
    client_id=os.getenv("AUTH_FLOW_CLIENT_ID"),
    client_secret=os.getenv("AUTH_FLOW_CLIENT_SECRET"),
    redirect_uri=os.getenv("REDIRECT_URI"),
    auth_url=os.getenv("AUTH_URL"),
    token_url=os.getenv("TOKEN_URL"),
    scope=os.getenv("SCOPE"),
)
auth_manager.authenticate()


All 6 env vars present ✓
Authenticating...
[DEBUG] authorization_url = https://idp.lanl.gov/as/authorization.oauth2?response_type=code&client_id=REDACTED&redirect_uri=https%3A%2F%2Fdatacatalog-d.lanl.gov%2Foauth%2F2.0%2FredirectURL.jsp&scope=den-datacat-adm&state=REDACTED
[DEBUG] page loaded: https://weblogin.lanl.gov/login
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
✓ Authentication successful


True

## Authenticate and run

In [7]:
auth_manager = OAuthManager(
    client_id=os.getenv("AUTH_FLOW_CLIENT_ID"),
    client_secret=os.getenv("AUTH_FLOW_CLIENT_SECRET"),
    redirect_uri=os.getenv("REDIRECT_URI"),
    auth_url=os.getenv("AUTH_URL"),
    token_url=os.getenv("TOKEN_URL"),
    scope=os.getenv("SCOPE"),
)
auth_manager.authenticate()

Authenticating...
[DEBUG] authorization_url = https://idp.lanl.gov/as/authorization.oauth2?response_type=code&client_id=REDACTED&redirect_uri=https%3A%2F%2Fdatacatalog-d.lanl.gov%2Foauth%2F2.0%2FredirectURL.jsp&scope=den-datacat-adm&state=REDACTED
[DEBUG] page loaded: https://weblogin.lanl.gov/login
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
✓ Authentication successful


True

In [9]:
sanity_check(auth_manager)
main(auth_manager)

view-details: 200
Sanity check passed -- OK to run main().
[plain] access_area_type -> HTTP 404 -> fixtures\plain.json
[doc_url] ap_1096_data_all -> HTTP 200 -> fixtures\doc_url.json
[access_role] affiliation_type_fvts -> HTTP 200 -> fixtures\access_role.json
[ods_link] announcement -> HTTP 200 -> fixtures\ods_link.json
[error_500] vlanlchangerequesttask -> HTTP 200 -> fixtures\error_500.json
[resource_rich] ben_prtn_elig_prfl_f -> HTTP 200 -> fixtures\resource_rich.json
[custom_tab] admin_option_type_fvts -> HTTP 200 -> fixtures\custom_tab.json

Done. Next: hand-write expected/<label>.py per the expected_plain_example.py template, then run contract_validator.py.


In [11]:
# --- Diagnostic A: is access_area_type gone from the list, or is view-details semantics changed?
r = auth_manager.get(BASE_URL + "/public/api/views", params={"serverId": SERVER_ID})
names = {(it.get("db") or it.get("databaseName"), it["name"]) for it in r.json()}
print("total views on post-migration dev:", len(names))
print("access_area_type present?", ("dataportal", "access_area_type") in names)

total views on post-migration dev: 4418
access_area_type present? False


In [12]:
# --- Diagnostic B: sweep ALL 12 vlanl*task views — is the 500 cohort gone entirely?
task_views = sorted(n for (db, n) in names if db == "dataportal" and n.startswith("vlanl") and "task" in n)
print(len(task_views), "vlanl*task views on the list")
for v in task_views:
    resp = auth_manager.get(BASE_URL + VIEW_DETAILS_PATH,
        params={"viewName": v, "databaseName": "dataportal", "serverId": SERVER_ID})
    print(f"  {v:<40} -> {resp.status_code}")

30 vlanl*task views on the list
  vlanlchangerequesttask                   -> 200
  vlanlconditiontask                       -> 200
  vlanlcractiontask                        -> 200
  vlanlcrcausetask                         -> 200
  vlanlenvironmentalwastetask              -> 200
  vlanlissuesmanagementtask                -> 200
  vlanlntsreporttask                       -> 200
  vlanlobservationtask                     -> 200
  vlanlopexsmereviewtask                   -> 200
  vlanlopextask                            -> 200
  vlanlpaaascreeningtask                   -> 200
  vlanlrailtasktrackerbase                 -> 200
  vlanlrailtasktrackercategorygridbase     -> 200
  vlanlrailtasktrackercategoryreferencemodulebase -> 200
  vlanlrailtasktrackercommentsbase         -> 200
  vlanlrailtasktrackercrossreferencecibase -> 200
  vlanlrailtasktrackercrossreferenceissuesmanagementbase -> 200
  vlanlrailtasktrackerlocationsbase        -> 200
  vlanlrailtasktrackerneedtoknowbase       -> 2

In [13]:
# --- Diagnostic C: how big is the drift? Compare list vs probe-era count
# (Step 0 pre-migration list had 15,014 raw / 15,007 active)
import collections
dbs = collections.Counter(db for (db, n) in names)
print(dbs)

Counter({'dataportal': 4414, 'ops_core_publication': 4})


In [14]:
# --- Re-select `plain`: intersect probe's zero-signal views with the CURRENT list, verify 200
import sqlite3
con = sqlite3.connect("probe_results.db"); con.row_factory = sqlite3.Row
zero = [r["view_name"] for r in con.execute(
    "SELECT view_name FROM processed WHERE n_signals=0 "
    "AND view_name NOT LIKE 'vlanl%task%' ORDER BY view_name")]
current = {n for (db, n) in names if db == "dataportal"}
for v in zero:
    if v in current:
        resp = auth_manager.get(BASE_URL + VIEW_DETAILS_PATH,
            params={"viewName": v, "databaseName": "dataportal", "serverId": SERVER_ID})
        if resp.status_code == 200:
            print("new plain fixture:", v); break

new plain fixture: inventory_daily_balance_fact


In [15]:
# --- Candidate for the new `second_vdb` fixture: list ops_core_publication's 4 views
for db, n in sorted(x for x in names if x[0] == "ops_core_publication"):
    resp = auth_manager.get(BASE_URL + VIEW_DETAILS_PATH,
        params={"viewName": n, "databaseName": db, "serverId": SERVER_ID})
    print(f"  {db}.{n} -> {resp.status_code}")

  ops_core_publication.i_ods_pa_cpnt_evthst -> 200
  ops_core_publication.i_px_lanl_curr_history_status -> 200
  ops_core_publication.p_ods_pa_cpnt_evthst_dims -> 200
  ops_core_publication.p_px_lanl_curr_qual_dims -> 200
